In [ ]:
## E_7172025_2 (online-video-cutter.com).mp4 - 188 birds


In [7]:
#!/usr/bin/env python3
# Complete Bird Detection Script

# Import all required libraries
import cv2
import numpy as np
import math
import tkinter as tk
from tkinter import filedialog, messagebox
import os
import time
from datetime import datetime

# ======================== IMPROVED DETECTION PARAMETERS ========================
FRAME_SKIP = 2  # Process every 2nd frame to avoid double counting
RESIZE_FACTOR = 0.8  # Resize for processing

# ROI parameters (percentage of frame height)
ROI_TOP_PERCENTAGE = 0  # Start analyzing from the top 0% (top of frame)
ROI_BOTTOM_PERCENTAGE = 100  # Stop at 50% of frame height (only analyze top half)

# Bird detection parameters
MIN_BIRD_AREA = 15
MAX_BIRD_AREA = 800
MIN_ASPECT_RATIO = 0.4
MAX_ASPECT_RATIO = 3.0

# Motion detection parameters
MOTION_THRESHOLD = 25  # Slightly lowered from original
MIN_MOTION_PIXELS = 5  # Minimum pixels that must be moving
MAX_STATIONARY_FRAMES = 10
MIN_MOVEMENT_DISTANCE = 15
MIN_FLIGHT_DURATION = 3
MAX_MATCHING_DISTANCE = 80

# New parameters for improved detection
CLOUD_TEXTURE_THRESHOLD = 20  # For cloud texture analysis
MIN_BIRD_SOLIDITY = 0.6  # Minimum solidity (area/convex hull area) for birds
MIN_BIRD_SPEED = 3  # Minimum speed (pixels/frame) for bird movement
MAX_BIRD_SPEED = 100  # Maximum speed for bird movement
DIRECTIONAL_CHANGE_THRESHOLD = 0.7  # For detecting non-linear movement (bird-like)
TEXTURE_NEIGHBORHOOD = 7  # Texture analysis kernel size
# ================================================================

prev_frame = None
prev_gray = None

def select_video_folder():
    """Open folder dialog to select folder containing video files"""
    root = tk.Tk()
    root.withdraw()
    
    folder_path = filedialog.askdirectory(
        title="Select Folder Containing Video Files"
    )
    
    root.destroy()
    return folder_path

def get_video_files(folder_path):
    """Get all video files from the specified folder"""
    video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv', '.webm', '.m4v', '.mpg', '.mpeg'}
    video_files = []
    
    if not os.path.exists(folder_path):
        return video_files
    
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if os.path.isfile(file_path):
            _, ext = os.path.splitext(filename.lower())
            if ext in video_extensions:
                video_files.append(file_path)
    
    return sorted(video_files)

def is_sky_color_improved(frame, bbox):
    """Improved sky color detection that better handles various sky conditions"""
    x, y, w, h = bbox
    frame_height, frame_width = frame.shape[:2]
    
    # Check if detection is within the ROI (Region of Interest)
    roi_top = int(frame_height * ROI_TOP_PERCENTAGE / 100)
    roi_bottom = int(frame_height * ROI_BOTTOM_PERCENTAGE / 100)
    
    # Skip if outside ROI
    if y < roi_top or y > roi_bottom:
        return False
    
    # Get region around detection
    border = 10
    x1 = max(0, x - border)
    y1 = max(0, y - border)
    x2 = min(frame_width, x + w + border)
    y2 = min(frame_height, y + h + border)
    
    region = frame[y1:y2, x1:x2]
    
    if region.size == 0:
        return False
    
    # Convert to HSV for better color analysis
    hsv_region = cv2.cvtColor(region, cv2.COLOR_BGR2HSV)
    
    # Sky color criteria - expanded to handle more sky conditions
    hue = hsv_region[:, :, 0]
    saturation = hsv_region[:, :, 1]
    value = hsv_region[:, :, 2]
    
    avg_saturation = np.mean(saturation)
    avg_value = np.mean(value)
    avg_hue = np.mean(hue)
    
    # White/Light grey sky (low saturation, high brightness)
    if avg_saturation < 60 and avg_value > 140:
        return True
    
    # Blue sky (hue in blue range, moderate saturation, good brightness)
    if 85 < avg_hue < 135 and avg_saturation < 160 and avg_value > 80:
        return True
    
    # Light blue/cyan sky
    if 75 < avg_hue < 105 and avg_saturation < 120 and avg_value > 100:
        return True
    
    # Grey/overcast sky (very low saturation, medium-high value)
    if avg_saturation < 30 and 80 < avg_value < 190:
        return True
    
    return False

def is_likely_cloud(frame, bbox):
    """Detect if a region is likely a cloud based on texture and color variance"""
    x, y, w, h = bbox
    
    # Minimum size check to avoid analyzing very small regions
    if w < 10 or h < 10:
        return False
    
    # Extract region
    region = frame[y:y+h, x:x+w]
    
    if region.size == 0:
        return False
    
    # Convert to grayscale for texture analysis
    gray_region = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
    
    # 1. Texture analysis (clouds have smooth texture)
    # Calculate local standard deviation as a measure of texture
    texture_kernel = np.ones((TEXTURE_NEIGHBORHOOD, TEXTURE_NEIGHBORHOOD), np.float32) / (TEXTURE_NEIGHBORHOOD * TEXTURE_NEIGHBORHOOD)
    local_mean = cv2.filter2D(gray_region.astype(np.float32), -1, texture_kernel)
    local_sqr_mean = cv2.filter2D(np.square(gray_region.astype(np.float32)), -1, texture_kernel)
    local_std = np.sqrt(local_sqr_mean - np.square(local_mean))
    
    # Clouds tend to have low texture variation (smooth)
    avg_texture = np.mean(local_std)
    
    # 2. Color variance (clouds have consistent color)
    hsv_region = cv2.cvtColor(region, cv2.COLOR_BGR2HSV)
    color_std = np.std(hsv_region[:,:,1])  # Saturation standard deviation
    
    # 3. Edge density (clouds have soft edges, birds have sharp silhouettes)
    edges = cv2.Canny(gray_region, 50, 150)
    edge_density = np.count_nonzero(edges) / region.size
    
    # Combined cloud detection criteria
    # Low texture + low color variance + low edge density = likely cloud
    if (avg_texture < CLOUD_TEXTURE_THRESHOLD and 
        color_std < 25 and 
        edge_density < 0.05):
        return True
    
    return False

def calculate_object_solidity(contour):
    """Calculate solidity (area/convex hull area) - birds typically have high solidity"""
    area = cv2.contourArea(contour)
    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    
    if hull_area > 0:
        return float(area) / hull_area
    return 0

def detect_motion_improved(current_detections, frame):
    """Improved motion detection that better distinguishes bird movement from clouds"""
    global prev_frame, prev_gray
    
    current_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    if prev_gray is None:
        prev_gray = current_gray
        return []  # No motion on first frame
    
    # Calculate optical flow for better motion analysis
    flow = cv2.calcOpticalFlowFarneback(prev_gray, current_gray, None, 
                                         0.5, 3, 15, 3, 5, 1.2, 0)
    
    # Simple frame difference as backup
    frame_diff = cv2.absdiff(prev_gray, current_gray)
    
    moving_birds = []
    for detection in current_detections:
        x, y, w, h = detection['bbox']
        
        # Skip if box extends outside frame
        if x < 0 or y < 0 or x+w >= frame.shape[1] or y+h >= frame.shape[0]:
            continue
            
        # 1. Extract motion region
        roi_diff = frame_diff[y:y+h, x:x+w]
        
        if roi_diff.size == 0:
            continue
        
        # 2. Count pixels with significant motion
        motion_pixels = np.count_nonzero(roi_diff > MOTION_THRESHOLD)
        
        # 3. Analyze optical flow in the region
        roi_flow = flow[y:y+h, x:x+w]
        
        # Skip empty regions
        if roi_flow.size == 0:
            continue
            
        # Calculate flow magnitude and direction
        mag, ang = cv2.cartToPolar(roi_flow[..., 0], roi_flow[..., 1])
        
        # Get average flow magnitude (speed)
        avg_magnitude = np.mean(mag)
        
        # Calculate flow direction consistency (birds change direction more than clouds)
        # Lower values mean more directional changes (more bird-like)
        if np.count_nonzero(mag > 0.5) > 10:  # Only if we have enough flow data
            flow_x = np.mean(roi_flow[..., 0])
            flow_y = np.mean(roi_flow[..., 1])
            flow_consistency = math.sqrt(flow_x**2 + flow_y**2) / (avg_magnitude + 1e-5)
        else:
            flow_consistency = 1.0  # Default: consistent direction
            
        # 4. Combine motion criteria
        # - Must have enough motion pixels
        # - Must have reasonable speed (not too slow, not too fast)
        # - For cloud-like objects, must have direction changes (inconsistent flow)
        if (motion_pixels >= MIN_MOTION_PIXELS and 
            MIN_BIRD_SPEED < avg_magnitude < MAX_BIRD_SPEED and 
            (flow_consistency < DIRECTIONAL_CHANGE_THRESHOLD or not is_likely_cloud(frame, detection['bbox']))):
            
            detection['motion_score'] = avg_magnitude
            detection['motion_pixels'] = motion_pixels
            detection['flow_consistency'] = flow_consistency
            moving_birds.append(detection)
    
    prev_gray = current_gray
    return moving_birds

def detect_birds_improved(frame):
    """Improved bird detection with enhanced shape and texture analysis"""
    # Resize frame
    height, width = frame.shape[:2]
    if RESIZE_FACTOR != 1.0:
        new_width = int(width * RESIZE_FACTOR)
        new_height = int(height * RESIZE_FACTOR)
        resized_frame = cv2.resize(frame, (new_width, new_height))
        scale_x = width / new_width
        scale_y = height / new_height
    else:
        resized_frame = frame
        scale_x = scale_y = 1.0
    
    # Convert to grayscale
    gray = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Use adaptive thresholding for better segmentation in varying conditions
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY_INV, 11, 2)
    
    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    detections = []
    for contour in contours:
        area = cv2.contourArea(contour)
        
        if MIN_BIRD_AREA < area < MAX_BIRD_AREA:
            x, y, w, h = cv2.boundingRect(contour)
            
            # Scale back to original size
            x = int(x * scale_x)
            y = int(y * scale_y)
            w = int(w * scale_x)
            h = int(h * scale_y)
            
            # Basic aspect ratio check
            aspect_ratio = w / h if h > 0 else 0
            if MIN_ASPECT_RATIO < aspect_ratio < MAX_ASPECT_RATIO:
                # Calculate solidity (area to convex hull ratio)
                # Birds tend to have compact shapes with good solidity
                solidity = calculate_object_solidity(contour)
                
                # Skip if solidity is too low (irregular shapes often not birds)
                if solidity < MIN_BIRD_SOLIDITY:
                    continue
                    
                # Check if it's likely a cloud
                if is_likely_cloud(frame, (x, y, w, h)):
                    continue
                
                center_x = x + w // 2
                center_y = y + h // 2
                
                detection = {
                    'bbox': (x, y, w, h),
                    'center': (center_x, center_y),
                    'area': area,
                    'solidity': solidity,
                    'aspect_ratio': aspect_ratio
                }
                detections.append(detection)
    
    return detections

class ImprovedBirdTracker:
    def __init__(self):
        self.tracks = {}
        self.track_id = 0
        self.confirmed_flying_birds = set()
        
    def update_tracks(self, detections):
        """Improved tracking with flight pattern analysis"""
        matched_tracks = {}
        current_frame_birds = []
        
        for detection in detections:
            x, y = detection['center']
            best_match = None
            min_distance = float('inf')
            
            # Find closest existing track
            for track_id, track_data in self.tracks.items():
                if track_id not in matched_tracks and len(track_data['positions']) > 0:
                    last_pos = track_data['positions'][-1]
                    distance = math.sqrt((x - last_pos[0])**2 + (y - last_pos[1])**2)
                    
                    if distance < min_distance and distance < MAX_MATCHING_DISTANCE:
                        min_distance = distance
                        best_match = track_id
            
            if best_match:
                # Update existing track
                self.tracks[best_match]['positions'].append((x, y))
                self.tracks[best_match]['stationary_count'] = 0
                
                # Store detection properties
                if 'detections' not in self.tracks[best_match]:
                    self.tracks[best_match]['detections'] = []
                self.tracks[best_match]['detections'].append(detection)
                
                matched_tracks[best_match] = detection
                
                # Check if bird qualifies as flying
                if self.is_flying_improved(best_match):
                    self.confirmed_flying_birds.add(best_match)
                    current_frame_birds.append(detection)
                    
            else:
                # Create new track
                self.tracks[self.track_id] = {
                    'positions': [(x, y)],
                    'stationary_count': 0,
                    'detections': [detection]
                }
                self.track_id += 1
        
        # Remove old tracks
        tracks_to_remove = []
        for track_id, track_data in self.tracks.items():
            if track_id not in matched_tracks:
                track_data['stationary_count'] += 1
                if track_data['stationary_count'] >= MAX_STATIONARY_FRAMES:
                    tracks_to_remove.append(track_id)
        
        for track_id in tracks_to_remove:
            del self.tracks[track_id]
        
        return current_frame_birds
    
    def is_flying_improved(self, track_id):
        """Improved flying detection with flight pattern analysis"""
        positions = self.tracks[track_id]['positions']
        
        if len(positions) < MIN_FLIGHT_DURATION:
            return False
        
        # 1. Calculate total movement distance
        total_distance = 0
        for i in range(1, len(positions)):
            prev_pos = positions[i-1]
            curr_pos = positions[i]
            distance = math.sqrt((curr_pos[0] - prev_pos[0])**2 + (curr_pos[1] - prev_pos[1])**2)
            total_distance += distance
        
        # 2. Calculate directionality (birds change direction)
        directions = []
        for i in range(1, len(positions)):
            prev_pos = positions[i-1]
            curr_pos = positions[i]
            dx = curr_pos[0] - prev_pos[0]
            dy = curr_pos[1] - prev_pos[1]
            
            if dx != 0 or dy != 0:  # Avoid division by zero
                angle = math.atan2(dy, dx)
                directions.append(angle)
        
        direction_changes = 0
        if len(directions) > 2:
            for i in range(1, len(directions)):
                diff = abs(directions[i] - directions[i-1])
                # Normalize to [0, π]
                if diff > math.pi:
                    diff = 2 * math.pi - diff
                
                # Count significant direction changes
                if diff > 0.3:  # ~17 degrees
                    direction_changes += 1
        
        # 3. Check flight characteristics from detections if available
        good_bird_features = 0
        if 'detections' in self.tracks[track_id] and len(self.tracks[track_id]['detections']) > 0:
            detections = self.tracks[track_id]['detections']
            
            # Average motion score (birds move consistently)
            if any('motion_score' in d for d in detections):
                motion_scores = [d.get('motion_score', 0) for d in detections if 'motion_score' in d]
                if motion_scores and np.mean(motion_scores) > MIN_BIRD_SPEED:
                    good_bird_features += 1
            
            # Flow consistency (birds change direction more than clouds)
            if any('flow_consistency' in d for d in detections):
                flow_consistencies = [d.get('flow_consistency', 1.0) for d in detections if 'flow_consistency' in d]
                if flow_consistencies and np.mean(flow_consistencies) < DIRECTIONAL_CHANGE_THRESHOLD:
                    good_bird_features += 1
                    
            # Aspect ratio variation (birds change shape during flight)
            if any('aspect_ratio' in d for d in detections):
                aspect_ratios = [d.get('aspect_ratio', 0) for d in detections if 'aspect_ratio' in d]
                if aspect_ratios and np.std(aspect_ratios) > 0.1:
                    good_bird_features += 1
        
        # Combined flight criteria
        min_flight_distance = MIN_MOVEMENT_DISTANCE
        
        # Criteria for confirmation:
        # 1. Must move minimum distance
        # 2. AND either: 
        #    a. Has at least one direction change
        #    b. OR has good bird-like motion features
        return (total_distance > min_flight_distance and 
                (direction_changes > 0 or good_bird_features >= 1))
    
    def get_unique_flying_birds_count(self):
        return len(self.confirmed_flying_birds)

def process_video_improved(video_path, show_display=False):
    """Improved video processing with better bird detection"""
    global prev_frame, prev_gray
    prev_frame = None
    prev_gray = None
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return None
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Processing: {os.path.basename(video_path)}")
    print(f"- Resolution: {width}x{height}, FPS: {fps}, Frames: {total_frames}")
    
    # Initialize tracking
    bird_tracker = ImprovedBirdTracker()
    
    # Statistics
    frame_count = 0
    max_concurrent_birds = 0
    start_time = time.time()
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            
            # Skip frames to avoid double counting
            if frame_count % FRAME_SKIP != 0:
                continue
            
            # Improved bird detection
            current_detections = detect_birds_improved(frame)
            
            # Filter by sky color
            sky_detections = []
            for detection in current_detections:
                if is_sky_color_improved(frame, detection['bbox']):
                    sky_detections.append(detection)
            
            # Improved motion detection
            moving_detections = detect_motion_improved(sky_detections, frame)
            
            # Update tracker with flight pattern analysis
            currently_flying_birds = bird_tracker.update_tracks(moving_detections)
            
            # Statistics
            unique_flying_birds = bird_tracker.get_unique_flying_birds_count()
            currently_active = len(currently_flying_birds)
            max_concurrent_birds = max(max_concurrent_birds, currently_active)
            
            # Display for single video mode
            if show_display:
                display_frame = frame.copy()
                
                # Draw ROI lines
                roi_top = int(height * ROI_TOP_PERCENTAGE / 100)
                roi_bottom = int(height * ROI_BOTTOM_PERCENTAGE / 100)
                cv2.line(display_frame, (0, roi_top), (width, roi_top), (255, 255, 0), 2)
                cv2.line(display_frame, (0, roi_bottom), (width, roi_bottom), (255, 0, 255), 2)
                
                # Draw detections
                for detection in current_detections:
                    x, y, w, h = detection['bbox']
                    cv2.rectangle(display_frame, (x, y), (x + w, y + h), (0, 255, 255), 1)
                
                # Draw sky-colored detections (cyan)
                for detection in sky_detections:
                    x, y, w, h = detection['bbox']
                    cv2.rectangle(display_frame, (x, y), (x + w, y + h), (255, 255, 0), 1)
                
                # Draw moving detections (green)
                for detection in moving_detections:
                    x, y, w, h = detection['bbox']
                    cv2.rectangle(display_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
                    motion_score = detection.get('motion_score', 0)
                    cv2.putText(display_frame, f"M:{motion_score:.1f}", (x, y - 5), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)
                
                # Draw confirmed flying birds (red)
                for detection in currently_flying_birds:
                    x, y, w, h = detection['bbox']
                    cv2.rectangle(display_frame, (x, y), (x + w, y + h), (0, 0, 255), 3)
                    cv2.putText(display_frame, "FLYING", (x, y - 20), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 2)
                
                # Information display
                cv2.putText(display_frame, f"Frame: {frame_count}/{total_frames} (every {FRAME_SKIP})", 
                           (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                
                cv2.putText(display_frame, f"UNIQUE Flying Birds: {unique_flying_birds}", 
                           (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                
                cv2.putText(display_frame, f"Detections: {len(current_detections)}", 
                           (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                
                cv2.putText(display_frame, f"Sky: {len(sky_detections)}", 
                           (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
                
                cv2.putText(display_frame, f"Moving: {len(moving_detections)}", 
                           (10, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
                
                cv2.putText(display_frame, f"Flying: {len(currently_flying_birds)}", 
                           (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                
                # Legend
                cv2.putText(display_frame, "Yellow: Top ROI | Magenta: Bottom ROI | Cyan: Initial | Green: Moving | Red: Flying", 
                           (10, height - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
                
                cv2.imshow("Improved Bird Counter", display_frame)
                
                key = cv2.waitKey(1) & 0xFF
                if key == ord('q') or key == 27:
                    break
            
            # Progress logging
            if frame_count % (100 * FRAME_SKIP) == 0:
                progress = frame_count / total_frames
                print(f"  Progress: {frame_count}/{total_frames} ({progress*100:.1f}%) - "
                      f"Birds: {unique_flying_birds} - "
                      f"Pipeline: {len(current_detections)}→{len(sky_detections)}→{len(moving_detections)}→{len(currently_flying_birds)}")
        
        # Get final results
        unique_flying_birds = bird_tracker.get_unique_flying_birds_count()
        total_time = time.time() - start_time
        
        print(f"  COMPLETED - Unique flying birds: {unique_flying_birds}")
        print(f"  Processing time: {total_time:.1f}s")
        
        return {
            'video_path': video_path,
            'video_name': os.path.basename(video_path),
            'total_frames': total_frames,
            'frames_processed': frame_count // FRAME_SKIP,
            'unique_flying_birds': unique_flying_birds,
            'max_concurrent_birds': max_concurrent_birds,
            'fps': fps,
            'duration_seconds': total_frames/fps if fps > 0 else 0,
            'processing_time': total_time
        }
        
    except Exception as e:
        print(f"  Error processing video: {str(e)}")
        return None
        
    finally:
        cap.release()
        if show_display:
            cv2.destroyAllWindows()

def write_results_to_file(results, output_file_path):
    """Write results to text file"""
    try:
        with open(output_file_path, 'w', encoding='utf-8') as f:
            f.write("IMPROVED BIRD COUNTING RESULTS\n")
            f.write("=" * 80 + "\n")
            f.write(f"Processing Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total Videos Processed: {len([r for r in results if r is not None])}\n")
            f.write("\nIMPROVED DETECTION SETTINGS:\n")
            f.write("- Enhanced Sky Color Detection: Works across multiple sky conditions\n")
            f.write("- Cloud Rejection: Texture and edge analysis to distinguish clouds\n")
            f.write("- Improved Motion Analysis: Using optical flow to detect bird-like movement\n")
            f.write("- Flight Pattern Analysis: Evaluates movement patterns typical of birds\n")
            f.write(f"- Frame Skip: Every {FRAME_SKIP} frames (no double counting)\n")
            f.write(f"- ROI: From {ROI_TOP_PERCENTAGE}% to {ROI_BOTTOM_PERCENTAGE}% of frame height\n")
            f.write("=" * 80 + "\n\n")
            
            # Summary statistics
            successful_results = [r for r in results if r is not None]
            if successful_results:
                total_birds = sum(r['unique_flying_birds'] for r in successful_results)
                avg_birds = total_birds / len(successful_results)
                max_birds = max(r['unique_flying_birds'] for r in successful_results)
                min_birds = min(r['unique_flying_birds'] for r in successful_results)
                
                f.write("SUMMARY STATISTICS:\n")
                f.write("-" * 40 + "\n")
                f.write(f"Total Birds Detected: {total_birds}\n")
                f.write(f"Average Birds per Video: {avg_birds:.1f}\n")
                f.write(f"Maximum Birds in Single Video: {max_birds}\n")
                f.write(f"Minimum Birds in Single Video: {min_birds}\n")
                f.write("\n")
            
            # Individual video results
            f.write("INDIVIDUAL VIDEO RESULTS:\n")
            f.write("-" * 40 + "\n")
            
            for i, result in enumerate(results, 1):
                if result is not None:
                    f.write(f"{i:2d}. {result['video_name']}\n")
                    f.write(f"    Flying Birds: {result['unique_flying_birds']}\n")
                    f.write(f"    Max Concurrent: {result['max_concurrent_birds']}\n")
                    f.write(f"    Duration: {result['duration_seconds']:.1f}s\n")
                    f.write(f"    Frames Processed: {result['frames_processed']} (every {FRAME_SKIP})\n")
                    f.write(f"    Processing Time: {result['processing_time']:.1f}s\n")
                    f.write("\n")
                else:
                    f.write(f"{i:2d}. [FAILED TO PROCESS]\n\n")
            
            # CSV format
            f.write("\nCSV FORMAT:\n")
            f.write("-" * 40 + "\n")
            f.write("Video_Name,Flying_Birds,Max_Concurrent,Duration_Seconds,Processing_Time\n")
            
            for result in results:
                if result is not None:
                    f.write(f"{result['video_name']},{result['unique_flying_birds']},"
                           f"{result['max_concurrent_birds']},{result['duration_seconds']:.1f},"
                           f"{result['processing_time']:.1f}\n")
        
        print(f"\nResults written to: {output_file_path}")
        return True
        
    except Exception as e:
        print(f"Error writing results to file: {str(e)}")
        return False

def select_video_file():
    """Open file dialog to select video file"""
    root = tk.Tk()
    root.withdraw()
    
    filetypes = [
        ("Video files", "*.mp4 *.avi *.mov *.mkv *.wmv *.flv *.webm"),
        ("All files", "*.*")
    ]
    
    filename = filedialog.askopenfilename(
        title="Select Video File",
        filetypes=filetypes
    )
    
    root.destroy()
    return filename

def adjust_roi_settings(name):
    top_percent =0
    if name.startswith("E"):
        ROI_BOTTOM_PERCENTAGE = 80
    elif name.startswith("D"):
        ROI_BOTTOM_PERCENTAGE = 75
    elif name.startswith("C"):
        ROI_BOTTOM_PERCENTAGE = 70
    else:
        ROI_BOTTOM_PERCENTAGE = 60
    return top_percent, ROI_BOTTOM_PERCENTAGE

def main():
    """Main function with improved bird detection"""
    global ROI_TOP_PERCENTAGE, ROI_BOTTOM_PERCENTAGE
    
    print("Improved Bird Counter")
    print("=" * 60)
    print("ENHANCED DETECTION FEATURES:")
    print("- Enhanced Sky Color Detection: Works across various sky conditions")
    print("- Cloud Rejection: Analyzes texture and edges to distinguish clouds from birds")
    print("- Improved Motion Analysis: Uses optical flow to detect bird-like movement")
    print("- Flight Pattern Analysis: Evaluates movement patterns typical of birds")
    print(f"- Frame processing: Every {FRAME_SKIP} frames (no double counting)")
    print(f"- ROI: Analyzing from {ROI_TOP_PERCENTAGE}% to {ROI_BOTTOM_PERCENTAGE}% of frame height")
    print("- Advanced tracking to prevent duplicate counting")
    print()
    
    
    # Ask user for processing mode
    print("Processing Options:")
    print("1. Batch process all videos in a folder")
    print("2. Process single video with display")
    print()
    
    while True:
        try:
            choice = input("Enter your choice (1 or 2): ").strip()
            if choice in ['1', '2']:
                break
            else:
                print("Please enter either 1 or 2")
        except KeyboardInterrupt:
            print("\nExiting...")
            return
    
    if choice == '2':
        # Single video processing
        video_path = select_video_file()
        
        if not video_path:
            print("No video file selected. Exiting...")
            return
        
        print(f"Selected video: {os.path.basename(video_path)}")
        ROI_TOP_PERCENTAGE, ROI_BOTTOM_PERCENTAGE = adjust_roi_settings(os.path.basename(video_path))
        try:
            result = process_video_improved(video_path, show_display=True)
            if result:
                print(f"\n{'='*50}")
                print(f"FINAL RESULTS")
                print(f"{'='*50}")
                print(f"Video: {result['video_name']}")
                print(f"FLYING BIRDS: {result['unique_flying_birds']}")
                print(f"{'='*50}")
        except Exception as e:
            print(f"Error processing video: {str(e)}")
        
        return
    
    # Batch processing mode
    print("\nBatch Processing Mode")
    
    folder_path = select_video_folder()
    if not folder_path:
        print("No folder selected. Exiting...")
        return
    
    video_files = get_video_files(folder_path)
    
    if not video_files:
        print(f"No video files found in folder: {folder_path}")
        return
    
    print(f"\nFound {len(video_files)} video files:")
    for i, video_file in enumerate(video_files, 1):
        print(f"  {i:2d}. {os.path.basename(video_file)}")
    
    # Confirm processing
    try:
        confirm = input(f"\nProcess all {len(video_files)} videos? (y/n): ").strip().lower()
        if confirm not in ['y', 'yes']:
            print("Processing cancelled.")
            return
    except KeyboardInterrupt:
        print("\nProcessing cancelled.")
        return
    
    # Process all videos
    results = []
    total_birds = 0
    successful_count = 0
    
    print(f"\n{'='*60}")
    print(f"STARTING BATCH PROCESSING")
    print(f"{'='*60}")
    
    start_time = datetime.now()
    
    for i, video_path in enumerate(video_files, 1):
        print(f"\n[{i}/{len(video_files)}] Processing: {os.path.basename(video_path)}")
        ROI_TOP_PERCENTAGE, ROI_BOTTOM_PERCENTAGE = adjust_roi_settings(os.path.basename(video_path))
        print("-" * 50)
        
        try:
            result = process_video_improved(video_path, show_display=False)
            
            if result is not None:
                results.append(result)
                total_birds += result['unique_flying_birds']
                successful_count += 1
            else:
                results.append(None)
                
        except Exception as e:
            print(f"  ERROR - {str(e)}")
            results.append(None)
    
    # Generate output file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    folder_name = os.path.basename(folder_path)
    output_filename = f"birds_count_improved_{folder_name}_{timestamp}.txt"
    output_path = os.path.join(folder_path, output_filename)
    
    # Write results
    write_results_to_file(results, output_path)
    
    # Final summary
    print(f"\n{'='*60}")
    print(f"BATCH PROCESSING COMPLETED")
    print(f"{'='*60}")
    print(f"Total Videos: {len(video_files)}")
    print(f"Successfully Processed: {successful_count}")
    print(f"TOTAL FLYING BIRDS: {total_birds}")
    if successful_count > 0:
        print(f"Average per Video: {total_birds/successful_count:.1f}")
    print(f"Results saved to: {output_filename}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

Improved Bird Counter
ENHANCED DETECTION FEATURES:
- Enhanced Sky Color Detection: Works across various sky conditions
- Cloud Rejection: Analyzes texture and edges to distinguish clouds from birds
- Improved Motion Analysis: Uses optical flow to detect bird-like movement
- Flight Pattern Analysis: Evaluates movement patterns typical of birds
- Frame processing: Every 3 frames (no double counting)
- ROI: Analyzing from 0% to 100% of frame height
- Advanced tracking to prevent duplicate counting

Processing Options:
1. Batch process all videos in a folder
2. Process single video with display

Selected video: C_7322025_1_output.avi
Processing: C_7322025_1_output.avi
- Resolution: 1920x1080, FPS: 29, Frames: 7615


/var/folders/0q/vsvgsd3s4ws83m1fcrysjbvc0000gn/T/ipykernel_3987/3661233208.py:154: RuntimeWarning: invalid value encountered in sqrt
  local_std = np.sqrt(local_sqr_mean - np.square(local_mean))


  Progress: 300/7615 (3.9%) - Birds: 55 - Pipeline: 261→165→56→0
  Progress: 600/7615 (7.9%) - Birds: 55 - Pipeline: 253→159→0→0
  Progress: 900/7615 (11.8%) - Birds: 55 - Pipeline: 239→157→0→0
  Progress: 1200/7615 (15.8%) - Birds: 55 - Pipeline: 221→134→0→0
  Progress: 1500/7615 (19.7%) - Birds: 76 - Pipeline: 206→115→0→0
  Progress: 1800/7615 (23.6%) - Birds: 78 - Pipeline: 214→126→0→0
  Progress: 2100/7615 (27.6%) - Birds: 80 - Pipeline: 179→120→0→0
  Progress: 2400/7615 (31.5%) - Birds: 82 - Pipeline: 227→124→0→0
  Progress: 2700/7615 (35.5%) - Birds: 124 - Pipeline: 207→116→0→0
  Progress: 3000/7615 (39.4%) - Birds: 125 - Pipeline: 203→112→0→0
  Progress: 3300/7615 (43.3%) - Birds: 143 - Pipeline: 222→119→0→0
  Progress: 3600/7615 (47.3%) - Birds: 143 - Pipeline: 219→124→0→0
  Progress: 3900/7615 (51.2%) - Birds: 144 - Pipeline: 215→123→0→0
  Progress: 4200/7615 (55.2%) - Birds: 145 - Pipeline: 230→139→0→0
  Progress: 4500/7615 (59.1%) - Birds: 145 - Pipeline: 212→114→0→0
  Progr